# 최적의 스티어링 레이어 찾기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/ko/05_layer_analysis.ipynb)

모든 레이어가 조향에 동일하게 적합한 것은 아닙니다. PSYCTL에는 SVM 기반 분리 분석을 사용하여 "positive"(성격이 드러나는)와 "neutral" 활성화를 가장 잘 구분하는 레이어를 순위화하는 레이어 분석 도구가 포함되어 있습니다.

**이 노트북에서 다루는 내용:**
1. 와일드카드 패턴을 사용하여 모든 MLP 레이어를 분석합니다
2. 순위 테이블(점수, 정확도, 마진)을 확인합니다
3. 레이어 분리 점수를 시각화합니다
4. 최상위 순위 레이어에서 벡터를 추출합니다

**요구사항:** Colab 무료 티어(T4 GPU). [HuggingFace 토큰](https://huggingface.co/settings/tokens).

**소요 시간:** 약 10분

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate datasets

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## 데이터셋 준비

분석을 위한 스티어링 데이터셋이 필요합니다. HuggingFace에서 커뮤니티 데이터셋을 다운로드합니다.

In [ ]:
from pathlib import Path
from datasets import load_dataset

ds = load_dataset("CaveduckAI/steer-personality-extroversion-ko", split="train")
dataset_dir = Path("./dataset_demo")
dataset_dir.mkdir(exist_ok=True)
dataset_path = dataset_dir / "data.jsonl"
ds.to_json(str(dataset_path))
print(f"Dataset: {len(ds)} samples saved to {dataset_path}")

## 레이어 분석 이론

"positive"와 "neutral" 텍스트를 모델에 입력하면, 각 레이어에서 내부 활성화가 달라집니다. 일부 레이어는 두 그룹 간에 명확한 분리를 보이고(조향에 적합), 다른 레이어는 활성화가 뒤섞여 있습니다(조향에 부적합).

**SVM 분석기**는 각 레이어에서 선형 SVM 분류기를 학습시켜 positive와 neutral 활성화를 분리한 후, 다음을 보고합니다:
- **점수(Score)**: 전반적인 분리 품질 (높을수록 좋음)
- **정확도(Accuracy)**: 분류 정확도 (SVM이 두 그룹을 얼마나 잘 분리하는지)
- **마진(Margin)**: SVM 마진 (마진이 클수록 분리가 강건함)

## 모든 MLP 레이어 분석

와일드카드 패턴 `model.layers[*].mlp`를 사용하여 모델의 모든 MLP 레이어를 분석합니다.

In [ ]:
from psyctl.core.layer_analyzer import LayerAnalyzer

MODEL_NAME = "google/gemma-3-270m-it"
LAYER_PATTERNS = ["model.layers[*].mlp"]

analyzer = LayerAnalyzer()

print(f"Analyzing all MLP layers of {MODEL_NAME}...")
results = analyzer.analyze_layers(
    model_name=MODEL_NAME,
    layers=LAYER_PATTERNS,
    dataset_path=dataset_path,
    method="svm",
    top_k=5,
    batch_size=8,
)

print(f"\nAnalyzed {results['total_layers']} layers")
print(f"\nTop 5 Layers:")
print(f"{'Rank':<6} {'Layer':<40} {'Score':>8} {'Accuracy':>10} {'Margin':>8}")
print("-" * 74)
for i, r in enumerate(results["rankings"][:5], 1):
    m = r["metrics"]
    print(f"{i:<6} {r['layer']:<40} {m.get('score', 0):>8.4f} {m.get('accuracy', 0):>10.4f} {m.get('margin', 0):>8.4f}")

## 시각화

In [ ]:
import matplotlib.pyplot as plt
import re

layers = []
scores = []
for r in results["rankings"]:
    layer_name = r["layer"]
    # Extract layer number for cleaner labels
    match = re.search(r"\[(\d+)\]", layer_name)
    label = f"Layer {match.group(1)}" if match else layer_name
    layers.append(label)
    scores.append(r["metrics"].get("score", 0))

# Highlight top 5
colors = ["#E85D75" if i < 5 else "#4A90D9" for i in range(len(layers))]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(layers)), scores, color=colors)
ax.set_xticks(range(len(layers)))
ax.set_xticklabels(layers, rotation=45, ha="right")
ax.set_ylabel("Separation Score")
ax.set_title(f"Layer Separation Scores — {MODEL_NAME}")
ax.axhline(y=scores[0] * 0.9, color="grey", linestyle="--", alpha=0.5, label="90% of best")
ax.legend()
plt.tight_layout()
plt.show()

## 최상위 레이어에서 추출

분석 결과를 활용하여 가장 높은 점수를 받은 레이어에서 벡터를 추출합니다.

In [ ]:
from psyctl.core.steering_extractor import SteeringExtractor
from psyctl.core.steering_applier import SteeringApplier

best_layer = results["top_k_layers"][0]
print(f"Best layer: {best_layer}")

extractor = SteeringExtractor()
output_path = Path("./vectors/best_layer.safetensors")

vectors = extractor.extract_steering_vector(
    model_name=MODEL_NAME,
    layers=[best_layer],
    dataset_path=dataset_path,
    output_path=output_path,
    method="mean_diff",
    normalize=True,
)

# Quick test
applier = SteeringApplier()
result = applier.apply_steering(
    model_name=MODEL_NAME,
    steering_vector_path=output_path,
    input_text="Hello, how are you?",
    strength=1.5,
    max_new_tokens=100,
    temperature=0.7,
)
print(f"\nSteered response (strength=1.5): {result}")